# Streamlined CRISPR Screen Analysis Tutorial

This tutorial walks through the Scanpy-style workflow provided by ``streamlined_crispr``.

## Prerequisites

Install the project in editable mode (with the optional ``test`` extras) to make the package importable and provide AnnData, NumPy, pandas, and SciPy:

```bash
pip install -e .[test]
```

## Prepare a demo dataset

The repository includes a small synthetic perturbation screen stored at ``data/demo_benchmark.h5ad``. Run ``python benchmarking/generate_demo_dataset.py`` to regenerate it if necessary.

In [ ]:
from pathlib import Path
import pandas as pd

import streamlined_crispr as scr
from benchmarking.generate_demo_dataset import write_demo_dataset

ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'demo_benchmark.h5ad'
OUTPUT_DIR = ROOT / 'notebooks' / 'tutorial_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    write_demo_dataset(DATA_PATH)

## Inspect the on-disk AnnData

``scr.read_h5ad_ondisk`` returns a read-only :class:`scr.AnnData` wrapper. The ``obs`` and ``var`` accessors expose ``head`` and ``load`` helpers so you can preview metadata without materialising the entire object.

In [ ]:
adata_ro = scr.read_h5ad_ondisk(DATA_PATH)
adata_ro

In [ ]:
adata_ro.obs.head()

In [ ]:
adata_ro.var.head()

## Quality control with ``scr.pp``

The preprocessing namespace mirrors Scanpy's API. Each function returns a new ``scr.AnnData`` object backed by an ``.h5ad`` file.

In [ ]:
adata_ro = scr.pp.qc_summary(
    adata_ro,
    perturbation_column='perturbation',
    min_genes=5,
    min_cells_per_perturbation=5,
    output_dir=OUTPUT_DIR,
    data_name='tutorial_filtered',
)
adata_ro.obs[['perturbation', 'qc_pass']].head()

## Pseudobulk aggregation with ``scr.pb``

Aggregate cells per perturbation on disk. The returned object can be inspected lazily, similar to the original dataset.

In [ ]:
adata_pb_ro = scr.pb.average_log_expression(
    adata_ro,
    perturbation_column='perturbation',
    output_dir=OUTPUT_DIR,
    data_name='tutorial_avg_log_expr',
)
adata_pb_ro.var.head()

## Differential expression with ``scr.tl``

Run a Wilcoxon test that mirrors ``scanpy.tl.rank_genes_groups``. ``scr.AnnData.uns`` entries also provide ``preview`` and ``load`` helpers.

In [ ]:
adata_ro = scr.tl.rank_genes_groups(
    adata_ro,
    perturbation_column='perturbation',
    method='wilcoxon',
    output_dir=OUTPUT_DIR,
    data_name='tutorial_wilcoxon',
)
adata_ro.uns['rank_genes_groups'].preview()

Use ``load`` to materialise the complete differential expression tables for downstream analysis.

In [ ]:
wilcoxon_full = adata_ro.uns['rank_genes_groups_full'].load()
group = wilcoxon_full['names'].dtype.names[0]
pd.DataFrame({
    'gene': wilcoxon_full['names'][group],
    'logfoldchange': wilcoxon_full['logfoldchanges'][group],
    'pval_adj': wilcoxon_full['pvals_adj'][group],
}).head()

The ``scr.AnnData`` handles close themselves when their Python objects go out of scope, so no explicit cleanup is required.